# Practical 2 - Representing Text as Numbers
### Bag of Words, TF-IDF, and N-grams


In [1]:
# 1. SETUP
!pip install -q scikit-learn nltk

import numpy as np
import pandas as pd
from collections import Counter
import re

import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

pd.set_option('display.max_columns', None)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


## 2. Toy Corpus

We'll use a tiny, deliberately simple set of sentences so we can see every number by eye before we scale up.

In [2]:
corpus = [
    "I binge watched the entire show on Netflix last night. It was the most exciting show I have seen in years.",
    "She streamed three movies on Netflix over the weekend. Netflix has a great collection of indie movies.",
    "We finished an entire web series on Amazon Prime in two days. Amazon Prime has really improved its original series.",
    "He prefers watching YouTube videos over Netflix shows. YouTube offers a wider variety of user-generated content.",
    "Cricket and football are the most popular sports in India. Most Indian fans follow cricket matches religiously.",
    "The football match went into a penalty shootout last night. It was a thrilling match for all the football fans.",
    "India won the cricket match by a huge margin. The Indian cricket team played exceptionally well today.",
    "Basketball is gaining popularity among college students. Many students spend their weekends playing basketball on campus.",
    "I do not like waking up early for morning classes. Morning classes are often very difficult to attend regularly.",
    "I really like staying up late to play video games. Playing video games is my favorite way to relax after a long day.",
    "Most students skip breakfast before their first lecture. Skipping breakfast can lead to low energy during the lecture.",
    "The college canteen gets crowded during lunch break. It is hard to find a seat in the canteen during the break.",
    "I listen to Spotify playlists while studying for exams. Spotify helps me focus better during intense studying sessions.",
    "She discovered a new artist through Instagram reels. Instagram reels are a great way to find trending music artists.",
    "Concerts and music festivals are fun to attend with friends. Attending a music festival with friends is an unforgettable experience."
]

for i, doc in enumerate(corpus):
    print(f"Doc {i}: {doc}")

Doc 0: I binge watched the entire show on Netflix last night. It was the most exciting show I have seen in years.
Doc 1: She streamed three movies on Netflix over the weekend. Netflix has a great collection of indie movies.
Doc 2: We finished an entire web series on Amazon Prime in two days. Amazon Prime has really improved its original series.
Doc 3: He prefers watching YouTube videos over Netflix shows. YouTube offers a wider variety of user-generated content.
Doc 4: Cricket and football are the most popular sports in India. Most Indian fans follow cricket matches religiously.
Doc 5: The football match went into a penalty shootout last night. It was a thrilling match for all the football fans.
Doc 6: India won the cricket match by a huge margin. The Indian cricket team played exceptionally well today.
Doc 7: Basketball is gaining popularity among college students. Many students spend their weekends playing basketball on campus.
Doc 8: I do not like waking up early for morning classes

## 3. Preprocessing

We'll lowercase, tokenize, and remove stopwords

In [3]:
stop_words = set(stopwords.words('english'))

def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    return tokens

processed_corpus = [preprocess(doc) for doc in corpus]

for i, doc in enumerate(processed_corpus):
    print(f"Doc {i}: {doc}")

Doc 0: ['binge', 'watched', 'entire', 'show', 'netflix', 'last', 'night', 'exciting', 'show', 'seen', 'years']
Doc 1: ['streamed', 'three', 'movies', 'netflix', 'weekend', 'netflix', 'great', 'collection', 'indie', 'movies']
Doc 2: ['finished', 'entire', 'web', 'series', 'amazon', 'prime', 'two', 'days', 'amazon', 'prime', 'really', 'improved', 'original', 'series']
Doc 3: ['prefers', 'watching', 'youtube', 'videos', 'netflix', 'shows', 'youtube', 'offers', 'wider', 'variety', 'content']
Doc 4: ['cricket', 'football', 'popular', 'sports', 'india', 'indian', 'fans', 'follow', 'cricket', 'matches', 'religiously']
Doc 5: ['football', 'match', 'went', 'penalty', 'shootout', 'last', 'night', 'thrilling', 'match', 'football', 'fans']
Doc 6: ['india', 'cricket', 'match', 'huge', 'margin', 'indian', 'cricket', 'team', 'played', 'exceptionally', 'well', 'today']
Doc 7: ['basketball', 'gaining', 'popularity', 'among', 'college', 'students', 'many', 'students', 'spend', 'weekends', 'playing', 'ba

## 4. Bag of Words

The idea: build a vocabulary (all unique words across all docs), then for each
document, count how many times each vocabulary word appears.

In [4]:
# Step 1: Build vocabulary
vocab = sorted(set(word for doc in processed_corpus for word in doc))
print("Vocabulary:", vocab)
print("Vocabulary size:", len(vocab))

Vocabulary: ['amazon', 'among', 'artist', 'artists', 'attend', 'attending', 'basketball', 'better', 'binge', 'break', 'breakfast', 'campus', 'canteen', 'classes', 'collection', 'college', 'concerts', 'content', 'cricket', 'crowded', 'day', 'days', 'difficult', 'discovered', 'early', 'energy', 'entire', 'exams', 'exceptionally', 'exciting', 'experience', 'fans', 'favorite', 'festival', 'festivals', 'find', 'finished', 'first', 'focus', 'follow', 'football', 'friends', 'fun', 'gaining', 'games', 'gets', 'great', 'hard', 'helps', 'huge', 'improved', 'india', 'indian', 'indie', 'instagram', 'intense', 'last', 'late', 'lead', 'lecture', 'like', 'listen', 'long', 'low', 'lunch', 'many', 'margin', 'match', 'matches', 'morning', 'movies', 'music', 'netflix', 'new', 'night', 'offers', 'often', 'original', 'penalty', 'play', 'played', 'playing', 'playlists', 'popular', 'popularity', 'prefers', 'prime', 'really', 'reels', 'regularly', 'relax', 'religiously', 'seat', 'seen', 'series', 'sessions', 

In [5]:
# Step 2: For each document, count word occurrences against the vocabulary
def bow_vector(doc_tokens, vocab):
    counts = Counter(doc_tokens)
    return [counts[word] for word in vocab]

bow_matrix_manual = [bow_vector(doc, vocab) for doc in processed_corpus]

bow_df_manual = pd.DataFrame(bow_matrix_manual, columns=vocab)
bow_df_manual.index = [f"Doc {i}" for i in range(len(corpus))]
bow_df_manual

,amazon,among,artist,artists,attend,attending,basketball,better,binge,break,breakfast,campus,canteen,classes,collection,college,concerts,content,cricket,crowded,day,days,difficult,discovered,early,energy,entire,exams,exceptionally,exciting,experience,fans,favorite,festival,festivals,find,finished,first,focus,follow,football,friends,fun,gaining,games,gets,great,hard,helps,huge,improved,india,indian,indie,instagram,intense,last,late,lead,lecture,like,listen,long,low,lunch,many,margin,match,matches,morning,movies,music,netflix,new,night,offers,often,original,penalty,play,played,playing,playlists,popular,popularity,prefers,prime,really,reels,regularly,relax,religiously,seat,seen,series,sessions,shootout,show,shows,skip,skipping,spend,sports,spotify,staying,streamed,students,studying,team,three,thrilling,today,trending,two,unforgettable,variety,video,videos,waking,watched,watching,way,web,weekend,weekends,well,went,wider,years,youtube
Doc 0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0
Doc 1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
Doc 2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,2,1,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
Doc 3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,1,0,2
Doc 4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Doc 5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
Doc 6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
Doc 7,0,1,0,0,0,0,2,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
Doc 8,0,0,0,0,1,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
Doc 9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,1,0,0,0,0,0,0,0,0


### Bag of Words - the sklearn way

In [6]:
from sklearn.feature_extraction.text import CountVectorizer

# Note: sklearn does its own tokenization + lowercasing.
# We pass the ORIGINAL corpus, and let stop_words='english' handle stopword removal.
vectorizer = CountVectorizer(stop_words='english')
bow_matrix_sklearn = vectorizer.fit_transform(corpus)

bow_df_sklearn = pd.DataFrame(
    bow_matrix_sklearn.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"Doc {i}" for i in range(len(corpus))]
)
bow_df_sklearn

,amazon,artist,artists,attend,attending,basketball,better,binge,break,breakfast,campus,canteen,classes,collection,college,concerts,content,cricket,crowded,day,days,difficult,discovered,early,energy,entire,exams,exceptionally,exciting,experience,fans,favorite,festival,festivals,finished,focus,follow,football,friends,fun,gaining,games,generated,gets,great,hard,helps,huge,improved,india,indian,indie,instagram,intense,late,lead,lecture,like,listen,long,low,lunch,margin,match,matches,morning,movies,music,netflix,new,night,offers,original,penalty,play,played,playing,playlists,popular,popularity,prefers,prime,really,reels,regularly,relax,religiously,seat,seen,series,sessions,shootout,shows,skip,skipping,spend,sports,spotify,staying,streamed,students,studying,team,thrilling,today,trending,unforgettable,user,variety,video,videos,waking,watched,watching,way,web,weekend,weekends,went,wider,won,years,youtube
Doc 0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0
Doc 1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
Doc 2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,2,1,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
Doc 3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,0,0,0,0,0,1,0,0,2
Doc 4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Doc 5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
Doc 6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
Doc 7,0,0,0,0,0,2,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
Doc 8,0,0,0,1,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
Doc 9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,1,0,0,0,0,0,0,0,0


## 5. TF-IDF

Formula:
- TF(word, doc) = (count of word in doc) / (total words in doc)
- IDF(word) = log( total docs / (1 + docs containing word) )
- TF-IDF = TF × IDF

In [7]:
import math

def compute_tf(doc_tokens):
    tf = {}
    total = len(doc_tokens)
    counts = Counter(doc_tokens)
    for word, count in counts.items():
        tf[word] = count / total
    return tf

def compute_idf(processed_corpus, vocab):
    N = len(processed_corpus)
    idf = {}
    for word in vocab:
        docs_containing = sum(1 for doc in processed_corpus if word in doc)
        idf[word] = math.log(N / (1 + docs_containing))
    return idf

tf_list = [compute_tf(doc) for doc in processed_corpus]
idf_dict = compute_idf(processed_corpus, vocab)

print("IDF values:")
for word in vocab:
    print(f"  {word}: {idf_dict[word]:.3f}")

IDF values:
  amazon: 2.015
  among: 2.015
  artist: 2.015
  artists: 2.015
  attend: 1.609
  attending: 2.015
  basketball: 2.015
  better: 2.015
  binge: 2.015
  break: 2.015
  breakfast: 2.015
  campus: 2.015
  canteen: 2.015
  classes: 2.015
  collection: 2.015
  college: 1.609
  concerts: 2.015
  content: 2.015
  cricket: 1.609
  crowded: 2.015
  day: 2.015
  days: 2.015
  difficult: 2.015
  discovered: 2.015
  early: 2.015
  energy: 2.015
  entire: 1.609
  exams: 2.015
  exceptionally: 2.015
  exciting: 2.015
  experience: 2.015
  fans: 1.609
  favorite: 2.015
  festival: 2.015
  festivals: 2.015
  find: 1.609
  finished: 2.015
  first: 2.015
  focus: 2.015
  follow: 2.015
  football: 1.609
  friends: 2.015
  fun: 2.015
  gaining: 2.015
  games: 2.015
  gets: 2.015
  great: 1.609
  hard: 2.015
  helps: 2.015
  huge: 2.015
  improved: 2.015
  india: 1.609
  indian: 1.609
  indie: 2.015
  instagram: 2.015
  intense: 2.015
  last: 1.609
  late: 2.015
  lead: 2.015
  lecture: 2.015
 

In [8]:
def compute_tfidf(tf, idf, vocab):
    return [tf.get(word, 0) * idf[word] for word in vocab]

tfidf_matrix_manual = [compute_tfidf(tf, idf_dict, vocab) for tf in tf_list]

tfidf_df_manual = pd.DataFrame(tfidf_matrix_manual, columns=vocab)
tfidf_df_manual.index = [f"Doc {i}" for i in range(len(corpus))]
tfidf_df_manual.round(3)

,amazon,among,artist,artists,attend,attending,basketball,better,binge,break,breakfast,campus,canteen,classes,collection,college,concerts,content,cricket,crowded,day,days,difficult,discovered,early,energy,entire,exams,exceptionally,exciting,experience,fans,favorite,festival,festivals,find,finished,first,focus,follow,football,friends,fun,gaining,games,gets,great,hard,helps,huge,improved,india,indian,indie,instagram,intense,last,late,lead,lecture,like,listen,long,low,lunch,many,margin,match,matches,morning,movies,music,netflix,new,night,offers,often,original,penalty,play,played,playing,playlists,popular,popularity,prefers,prime,really,reels,regularly,relax,religiously,seat,seen,series,sessions,shootout,show,shows,skip,skipping,spend,sports,spotify,staying,streamed,students,studying,team,three,thrilling,today,trending,two,unforgettable,variety,video,videos,waking,watched,watching,way,web,weekend,weekends,well,went,wider,years,youtube
Doc 0,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.183,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.146,0.000,0.000,0.183,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.146,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.120,0.000,0.146,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.183,0.000,0.000,0.000,0.366,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.183,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.183,0.000
Doc 1,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.201,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.161,0.000,0.000,0.000,0.000,0.000,0.000,0.201,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.403,0.000,0.264,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.201,0.000,0.000,0.000,0.201,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.201,0.000,0.000,0.000,0.000,0.000,0.000
Doc 2,0.288,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.144,0.000,0.000,0.000,0.000,0.115,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.144,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.144,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.144,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.288,0.115,0.00,0.000,0.000,0.000,0.000,0.000,0.288,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.144,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.144,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Doc 3,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.183,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.120,0.000,0.000,0.183,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.183,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.183,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.183,0.000,0.

- Find a word that appears in ALL docs. What's its IDF? Why does that make sense?

### TF-IDF - the sklearn way

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix_sklearn = tfidf_vectorizer.fit_transform(corpus)

tfidf_df_sklearn = pd.DataFrame(
    tfidf_matrix_sklearn.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f"Doc {i}" for i in range(len(corpus))]
)
tfidf_df_sklearn.round(3)

,amazon,artist,artists,attend,attending,basketball,better,binge,break,breakfast,campus,canteen,classes,collection,college,concerts,content,cricket,crowded,day,days,difficult,discovered,early,energy,entire,exams,exceptionally,exciting,experience,fans,favorite,festival,festivals,finished,focus,follow,football,friends,fun,gaining,games,generated,gets,great,hard,helps,huge,improved,india,indian,indie,instagram,intense,late,lead,lecture,like,listen,long,low,lunch,margin,match,matches,morning,movies,music,netflix,new,night,offers,original,penalty,play,played,playing,playlists,popular,popularity,prefers,prime,really,reels,regularly,relax,religiously,seat,seen,series,sessions,shootout,shows,skip,skipping,spend,sports,spotify,staying,streamed,students,studying,team,thrilling,today,trending,unforgettable,user,variety,video,videos,waking,watched,watching,way,web,weekend,weekends,went,wider,won,years,youtube
Doc 0,0.000,0.000,0.000,0.000,0.00,0.000,0.00,0.375,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.00,0.326,0.00,0.000,0.375,0.00,0.000,0.000,0.00,0.00,0.000,0.00,0.000,0.000,0.00,0.00,0.000,0.000,0.000,0.00,0.000,0.00,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.00,0.000,0.000,0.00,0.000,0.00,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.291,0.000,0.326,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.375,0.000,0.00,0.000,0.000,0.00,0.00,0.000,0.000,0.0,0.000,0.000,0.000,0.0,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.375,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.375,0.000
Doc 1,0.000,0.000,0.000,0.000,0.00,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.299,0.000,0.00,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.00,0.000,0.000,0.00,0.000,0.000,0.00,0.00,0.000,0.00,0.000,0.000,0.00,0.00,0.000,0.000,0.000,0.00,0.260,0.00,0.00,0.000,0.000,0.000,0.000,0.299,0.000,0.00,0.000,0.00,0.000,0.000,0.00,0.000,0.00,0.00,0.000,0.000,0.000,0.000,0.599,0.000,0.464,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.00,0.000,0.000,0.00,0.00,0.000,0.000,0.0,0.000,0.299,0.000,0.0,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.299,0.000,0.000,0.000,0.000,0.000,0.000
Doc 2,0.465,0.000,0.000,0.000,0.00,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.00,0.000,0.232,0.000,0.000,0.000,0.00,0.202,0.00,0.000,0.000,0.00,0.000,0.000,0.00,0.00,0.232,0.00,0.000,0.000,0.00,0.00,0.000,0.000,0.000,0.00,0.000,0.00,0.00,0.000,0.232,0.000,0.000,0.000,0.000,0.00,0.000,0.00,0.000,0.000,0.00,0.000,0.00,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.232,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.465,0.202,0.000,0.000,0.000,0.000,0.00,0.000,0.465,0.00,0.000,0.000,0.00,0.00,0.000,0.000,0.0,0.000,0.000,0.000,0.0,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.232,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Doc 3,0.000,0.000,0.000,0.000,0.00,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.262,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.00,0.000,0.000,0.00,0.000,0.000,0.00,0.00,0.000,0.00,0.000,0.000,0.00,0.00,0.000,0.000,0.262,0.00,0.000,0.00,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.00,0.000,0.000,0.00,0.000,0.00,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.203,0.000,0.000,0.262,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.262,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.00,0.000,0.262,0.00,0.00,0.000,0.000,0.0,0.000,0.000,0.000,0.0,0.000,0.000,0.000,0.000,0.00,0.262,0.262,0.000,0.262,0.000,0.000,0.262,0.000,0.000,0.000,0.000,0.000,0.262,0.000,0.000,0.523
Doc 4,0.000,0.000,0.000,0.000,0.00,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.523,0.00,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.00,0.000,0.000,0.00,0.261,0.000,0.00,0.00,0.000,0.00,0.301,0.261,0.00,0.00,

## 6. N-grams

Unigrams lose word order entirely. Let's fix that partially with bigrams/trigrams.

In [10]:
from sklearn.feature_extraction.text import CountVectorizer

# unigrams only (what we've done so far)
uni_vectorizer = CountVectorizer(stop_words='english', ngram_range=(1,1))
uni_matrix = uni_vectorizer.fit_transform(corpus)
print("Unigram features:", uni_vectorizer.get_feature_names_out())

# bigrams only
bi_vectorizer = CountVectorizer(stop_words='english', ngram_range=(2,2))
bi_matrix = bi_vectorizer.fit_transform(corpus)
print("Bigram features:", bi_vectorizer.get_feature_names_out())

print()

# unigrams + bigrams together (most common real-world choice)
uni_bi_vectorizer = CountVectorizer(stop_words='english', ngram_range=(1,2))
uni_bi_matrix = uni_bi_vectorizer.fit_transform(corpus)
print("Unigram+Bigram feature count:", len(uni_bi_vectorizer.get_feature_names_out()))

Unigram features: ['amazon' 'artist' 'artists' 'attend' 'attending' 'basketball' 'better'
 'binge' 'break' 'breakfast' 'campus' 'canteen' 'classes' 'collection'
 'college' 'concerts' 'content' 'cricket' 'crowded' 'day' 'days'
 'difficult' 'discovered' 'early' 'energy' 'entire' 'exams'
 'exceptionally' 'exciting' 'experience' 'fans' 'favorite' 'festival'
 'festivals' 'finished' 'focus' 'follow' 'football' 'friends' 'fun'
 'gaining' 'games' 'generated' 'gets' 'great' 'hard' 'helps' 'huge'
 'improved' 'india' 'indian' 'indie' 'instagram' 'intense' 'late' 'lead'
 'lecture' 'like' 'listen' 'long' 'low' 'lunch' 'margin' 'match' 'matches'
 'morning' 'movies' 'music' 'netflix' 'new' 'night' 'offers' 'original'
 'penalty' 'play' 'played' 'playing' 'playlists' 'popular' 'popularity'
 'prefers' 'prime' 'really' 'reels' 'regularly' 'relax' 'religiously'
 'seat' 'seen' 'series' 'sessions' 'shootout' 'shows' 'skip' 'skipping'
 'spend' 'sports' 'spotify' 'staying' 'streamed' 'students' 'studying'
 't

In [11]:
# The "not good" problem
demo = ["I like this movie", "I do not like this movie"]

uni_demo = CountVectorizer(ngram_range=(1,1))
print("Unigrams:\n", pd.DataFrame(uni_demo.fit_transform(demo).toarray(),
                                    columns=uni_demo.get_feature_names_out()))

bi_demo = CountVectorizer(ngram_range=(2,2))
print("\nBigrams:\n", pd.DataFrame(bi_demo.fit_transform(demo).toarray(),
                                     columns=bi_demo.get_feature_names_out()))

Unigrams:
    do  like  movie  not  this
0   0     1      1    0     1
1   1     1      1    1     1

Bigrams:
    do not  like this  not like  this movie
0       0          1         0           1
1       1          1         1           1


**Discuss:** With unigrams alone, "like" appears in both sentences equally - a model
might think both are positive. What does the bigram table capture that the unigram
table completely misses?

## 7. Putting it together - Document Similarity

Now let's use TF-IDF vectors for something useful: finding which documents are
most similar to each other, using cosine similarity.

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix_sklearn)

sim_df = pd.DataFrame(
    similarity_matrix,
    index=[f"Doc {i}" for i in range(len(corpus))],
    columns=[f"Doc {i}" for i in range(len(corpus))]
)
sim_df.round(2)

,Doc 0,Doc 1,Doc 2,Doc 3,Doc 4,Doc 5,Doc 6,Doc 7,Doc 8,Doc 9,Doc 10,Doc 11,Doc 12,Doc 13,Doc 14
Doc 0,1.00,0.13,0.07,0.06,0.00,0.08,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00
Doc 1,0.13,1.00,0.00,0.09,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.06,0.00
Doc 2,0.07,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.04,0.00,0.00,0.0,0.00,0.00
Doc 3,0.06,0.09,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00
Doc 4,0.00,0.00,0.00,0.00,1.00,0.20,0.39,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00
Doc 5,0.08,0.00,0.00,0.00,0.20,1.00,0.13,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00
Doc 6,0.00,0.00,0.00,0.00,0.39,0.13,1.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00
Doc 7,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.05,0.11,0.06,0.0,0.00,0.00
Doc 8,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.05,0.00,0.00,0.0,0.00,0.05
Doc 9,0.00,0.00,0.04,0.00,0.00,0.00,0.00,0.05,0.05,1.00,0.00,0.00,0.0,0.05,0.00


In [13]:
# Which two DIFFERENT documents are most similar?
np.fill_diagonal(similarity_matrix, 0)  # ignore self-similarity
i, j = np.unravel_index(similarity_matrix.argmax(), similarity_matrix.shape)
print(f"Most similar pair: Doc {i} and Doc {j}")
print(f"  '{corpus[i]}'")
print(f"  '{corpus[j]}'")
print(f"  Similarity score: {similarity_matrix[i][j]:.3f}")

Most similar pair: Doc 4 and Doc 6
  'Cricket and football are the most popular sports in India. Most Indian fans follow cricket matches religiously.'
  'India won the cricket match by a huge margin. The Indian cricket team played exceptionally well today.'
  Similarity score: 0.389


# Bonus Implementation with real data to show the usecase


In the final section, we demonstrate a real-world application of text representation: **Authorship Attribution**.

We move beyond simple unigrams to use **Character N-grams** (2-4 characters) and **TF-IDF**. By chunking long texts from Gutenberg into smaller pieces, we allowed the TF-IDF algorithm to effectively identify unique stylistic signatures. This approach successfully identified the 'Mystery Passage' as being written by **Arthur Conan Doyle** by comparing the average cosine similarity across all known chunks of work from multiple authors.

In [14]:
import urllib.request
import re

gutenberg_urls = {
    "Arthur Conan Doyle": "https://www.gutenberg.org/files/1661/1661-0.txt",
    "Jane Austen":        "https://www.gutenberg.org/files/1342/1342-0.txt",
    "Edgar Allan Poe":    "https://www.gutenberg.org/cache/epub/2148/pg2148.txt",
    "Charles Dickens":    "https://www.gutenberg.org/files/1400/1400-0.txt",
}

def clean_gutenberg_text(text):
    text = text.replace('\r\n', ' ').replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def fetch_text(url, start_offset=5000, length=15000):
    """Fetch a much larger chunk, skip Gutenberg's header more safely."""
    with urllib.request.urlopen(url) as response:
        raw = response.read().decode('utf-8', errors='ignore')

    # Try to find the actual start of content, skipping Gutenberg's license header
    marker = "*** START OF"
    start_idx = raw.find(marker)
    if start_idx != -1:
        # jump past the marker line itself
        start_idx = raw.find('\n', start_idx) + 1
    else:
        start_idx = start_offset  # fallback

    chunk = raw[start_idx + start_offset : start_idx + start_offset + length]
    return clean_gutenberg_text(chunk)

# Fetch MUCH larger training samples per author — ~15,000 characters (~2500-3000 words)
author_samples = {}
for author, url in gutenberg_urls.items():
    author_samples[author] = fetch_text(url, start_offset=3000, length=15000)
    print(f"{author}: {len(author_samples[author])} characters, "
          f"~{len(author_samples[author].split())} words")

Arthur Conan Doyle: 14549 characters, ~2633 words
Jane Austen: 14992 characters, ~2550 words
Edgar Allan Poe: 13193 characters, ~2288 words
Charles Dickens: 14611 characters, ~2762 words


### From Data Loading to Mystery Selection
After fetching approximately 15,000 characters for our four target authors, we now have a representative 'baseline' of their writing styles. Next, we will pull a completely different, non-overlapping section from Sir Arthur Conan Doyle's work to act as our 'Mystery Passage' for testing.

In [15]:
# Mystery passage: same Doyle book, a DIFFERENT and clearly non-overlapping chunk
# Push the offset far enough forward that it can't overlap the training chunk above
mystery_passage = fetch_text(
    gutenberg_urls["Arthur Conan Doyle"],
    start_offset=25000,   # well past the 3000-18000 range used for training
    length=2000            # bigger than before too — more signal to match on
)

print("Mystery passage preview:\n")
print(mystery_passage[:300])

Mystery passage preview:

h these details, but I have to let you see my little difficulties, if you are to understand the situation.” “I am following you closely,” I answered. “I was still balancing the matter in my mind when a hansom cab drove up to Briony Lodge, and a gentleman sprang out. He was a remarkably handsome man,


### Evaluating Unigram Performance
We've successfully isolated a mystery passage from *A Scandal in Bohemia*. Now, let's see if a basic TF-IDF approach using word-level unigrams and bigrams can correctly identify the author. This approach relies on the specific vocabulary used by the writers.

We try with unigram words

In [16]:
authors = list(author_samples.keys())
known_texts = list(author_samples.values())

vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=1000)
all_texts = known_texts + [mystery_passage]
tfidf_matrix = vectorizer.fit_transform(all_texts)

mystery_vector = tfidf_matrix[-1]
author_vectors = tfidf_matrix[:-1]
scores = cosine_similarity(mystery_vector, author_vectors).flatten()

print("🕵️ Style match scores:\n")
for author, score in sorted(zip(authors, scores), key=lambda x: -x[1]):
    bar = "█" * int(score * 50)
    print(f"{author:20s} {score:.3f} {bar}")

print(f"\n🎯 Verdict: {authors[np.argmax(scores)]}")
print(f"✅ Correct answer: Arthur Conan Doyle")

🕵️ Style match scores:

Charles Dickens      0.126 ██████
Arthur Conan Doyle   0.119 █████
Edgar Allan Poe      0.048 ██
Jane Austen          0.034 █

🎯 Verdict: Charles Dickens
✅ Correct answer: Arthur Conan Doyle


### Moving to Character-Level Analysis
The previous word-level model incorrectly guessed Charles Dickens. This often happens because single documents (especially small chunks) might share similar topics or common words by chance. To fix this, we'll try analyzing sub-word patterns (Character N-grams), which capture 'fingerprints' like suffix usage and punctuation habits.

### Refining with Character N-Grams
We just tried a basic character n-gram model, but it still favored Dickens, likely because the 'training' sample for each author was just one single large block. Next, we will implement **Document Chunking**. By splitting each author's work into many smaller pieces, we allow the TF-IDF algorithm to distinguish between what is common across an author's entire style versus what is just unique to a specific paragraph.

In [17]:
vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 4),
    max_features=2000
)

all_texts = known_texts + [mystery_passage]
tfidf_matrix = vectorizer.fit_transform(all_texts)

mystery_vector = tfidf_matrix[-1]
author_vectors = tfidf_matrix[:-1]
scores = cosine_similarity(mystery_vector, author_vectors).flatten()

print("🕵️ Style match scores (character n-grams):\n")
for author, score in sorted(zip(authors, scores), key=lambda x: -x[1]):
    bar = "█" * int(score * 50)
    print(f"{author:20s} {score:.3f} {bar}")

print(f"\n🎯 Verdict: {authors[np.argmax(scores)]}")
print(f"✅ Correct answer: Arthur Conan Doyle")

🕵️ Style match scores (character n-grams):

Charles Dickens      0.935 ██████████████████████████████████████████████
Arthur Conan Doyle   0.923 ██████████████████████████████████████████████
Jane Austen          0.903 █████████████████████████████████████████████
Edgar Allan Poe      0.896 ████████████████████████████████████████████

🎯 Verdict: Charles Dickens
✅ Correct answer: Arthur Conan Doyle


### Creating the Chunked Dataset
Our previous attempt showed that character n-grams are powerful but need better data distribution. We've now written a function to break the texts into 1,500-character segments. Next, we will re-run the TF-IDF vectorizer on this chunked library and calculate the average similarity for the mystery passage across all chunks belonging to each author.

In [18]:
def chunk_text(text, chunk_size=1500):
    """Split into multiple chunks so TF-IDF has enough docs to compute real IDF."""
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size) if len(text[i:i+chunk_size]) > 500]

# Fetch bigger raw text per author (so we have enough to chunk from)
author_samples_raw = {}
for author, url in gutenberg_urls.items():
    author_samples_raw[author] = fetch_text(url, start_offset=3000, length=30000)

# Build a labeled dataset: many chunks per author
chunk_texts = []
chunk_labels = []

for author, text in author_samples_raw.items():
    chunks = chunk_text(text, chunk_size=1500)
    chunk_texts.extend(chunks)
    chunk_labels.extend([author] * len(chunks))
    print(f"{author}: {len(chunks)} chunks")

print(f"\nTotal training chunks: {len(chunk_texts)}")

Arthur Conan Doyle: 20 chunks
Jane Austen: 18 chunks
Edgar Allan Poe: 18 chunks
Charles Dickens: 20 chunks

Total training chunks: 76


### Final Prediction: The Power of Chunks and Sub-words
With 76 distinct training chunks, the TF-IDF calculation finally has enough 'documents' to accurately penalize generic patterns and highlight author-specific styles. Let's look at the final results to see if Doyle is correctly identified.

In [19]:
# Vectorize all chunks together — NOW TF-IDF has enough documents for IDF to mean something
vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 4),
    max_features=2000,
    min_df=2,        # ignore n-grams that appear in fewer than 2 chunks (too rare/noisy)
    max_df=0.8        # ignore n-grams that appear in >80% of chunks (too generic — "th", " a", etc.)
)

all_texts = chunk_texts + [mystery_passage]
tfidf_matrix = vectorizer.fit_transform(all_texts)

mystery_vector = tfidf_matrix[-1]
chunk_vectors = tfidf_matrix[:-1]

scores = cosine_similarity(mystery_vector, chunk_vectors).flatten()

# Average the similarity score per author across all their chunks
import pandas as pd
score_df = pd.DataFrame({'author': chunk_labels, 'score': scores})
author_avg_scores = score_df.groupby('author')['score'].mean().sort_values(ascending=False)

print("🕵️ Style match scores (averaged across chunks):\n")
for author, score in author_avg_scores.items():
    bar = "█" * int(score * 50)
    print(f"{author:20s} {score:.3f} {bar}")

print(f"\n🎯 Verdict: {author_avg_scores.idxmax()}")
print(f"✅ Correct answer: Arthur Conan Doyle")

🕵️ Style match scores (averaged across chunks):

Arthur Conan Doyle   0.415 ████████████████████
Charles Dickens      0.356 █████████████████
Edgar Allan Poe      0.272 █████████████
Jane Austen          0.258 ████████████

🎯 Verdict: Arthur Conan Doyle
✅ Correct answer: Arthur Conan Doyle
